# 4. Physical stage from saved warmup
Load notebook 3's immutable warmup and exact inputs. Disable warmup and start physical factorization from those fields. Each outer round alternates ER updates with a shared charge/magnetization projection; set OUTER_ITERATIONS=0 for a **single final physical projection without detector updates**.

Saturation anchors are optional: list scan indices independently known to be saturated and their signs. Without two opposite saturated anchors the fitted magnetization is relative contrast, not an absolute calibrated magnetization. The material region defaults to the largest support component saved in notebook 3. Illumination and phase switches are independent.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
if HERE.name != "permanent_magnets":
    HERE = HERE / "permanent_magnets"
REPO = HERE.parent
sys.path.insert(0, str(REPO))
from permanent_magnets import workflow

from library import phase_retrieval_universal as universal

In [ ]:
PROCESSED = Path("/path/to/processed")  # Edit: same output folder in all four notebooks.
INPUT = PROCESSED / "03_warmup.h5"
OUTPUT = PROCESSED / "04_physical.h5"
SATURATED_SCAN_INDICES = []  # Example: [1, 2], only when independently known to saturate.
SATURATION_SIGNS = []        # Corresponding +1 or -1 values.
OUTER_ITERATIONS = 10        # 0: projection only; >0: ER + physical projection rounds.
INNER_ITERATIONS = 50
PHYSICAL_ITERATIONS = 50
ALIGN_GLOBAL_PHASE = True
FIT_ILLUMINATION_INTENSITY = True
FIT_APERTURE_ILLUMINATION = True
OBJECT_ILLUMINATION_MODE = "linear_plane"  # RH-based gain extrapolation to object.
APERTURE_FACTOR_TYPE = "complex"
COMMON_SECONDARY_MODES = True
INITIAL_BINARY_MAGNETIZATION = False  # Continuous measured-contrast initializer by default.
CLIP_MAGNETIZATION = False
MAGNETIZATION_TREND = "none"  # none, increasing, decreasing; observation order.
PHYSICAL_SPATIAL_WEIGHTING = "exit_wave_intensity"  # None disables spatial weighting.
DETECTOR_PIXEL_SIZE_M = None  # Needed only for nonzero focus propagation.
DETECTOR_DISTANCE_M = None
FOCUS_PROP_UM = 0.0
FOCUS_PHASE_RAD = 0.0
FFT_WORKERS = 1

# Remaining explicit solver settings (same convention as the controls above).
PROJECTION_MODEL = "physical_factorized"
RECENTER_MODAL_SUPPORTS = True
MODE_SUPPORT_CENTER = "image"
CROP = 0
BINNING = 1
ROI = None
HOLOGRAM_INTENSITY_CUTOFF_VMIN = -1
WARMUP_NIT = 0
WARMUP_REFERENCE_MODE = None
WARMUP_REFERENCE_NIT = None
WARMUP_OTHER_MODE = None
WARMUP_OTHER_NIT = None
PARTIAL_COHERENCE = False
INNER_MODE = "ER"
BETA_MODE = "const"
BETA_ZERO = .5
ALPHA_ZERO = .5
TV_FREQ = 10
AVERAGE_IMG = 1
FIT_MATERIAL_THICKNESS = False
PHYSICAL_PROJECTION_OBJECT_ROI = True
PHYSICAL_PHASE_REFERENCE = True
BINARY_MAGNETIZATION = False
PROJECTION_RELAXATION = 1.
FINAL_PROJECTION_RELAXATION = 1.
FINAL_FOURIER_CONSTRAINT = False
PROJECTION_CONSTRAINTS_INSIDE_SUPPORT_ONLY = True
ZERO_MAGNETIZATION_OUTSIDE_SUPPORT = True
RANDOM_SEED = 0

# Modes, material, anchors, focus geometry, and projection frequency are derived
# from the saved warmup inputs below, rather than entered independently.


In [ ]:
with h5py.File(INPUT, "r") as h:
    warmup = h["warmup_fields"][()]
    holograms = h["holograms"][()]
    mask_pixel = h["mask_pixel"][()].astype(bool)
    support = h["supportmask"][()]
    material = h["material_thickness"][()]
    modes = h["modes"][()].tolist()
    scan_indices = h["scan_indices"][()]
    energies = h["energy_eV"][()]
    polarizations = h["polarization"][()]
    state_labels = h["state_labels"].asstr()[()].tolist()
    assert h.attrs["crop"] == 0 and h.attrs["binning"] == 1
assert len(SATURATED_SCAN_INDICES) == len(SATURATION_SIGNS)
assert len(set(SATURATED_SCAN_INDICES)) == len(SATURATED_SCAN_INDICES)
assert set(SATURATED_SCAN_INDICES).issubset(set(scan_indices))
assert all(sign in (-1, 1) for sign in SATURATION_SIGNS)
saturated_states = {f"scan_{index}": float(sign) for index, sign in zip(SATURATED_SCAN_INDICES, SATURATION_SIGNS)}
calibrated = set(SATURATION_SIGNS) == {-1, 1}
print("Calibration:", "two saturated endpoints" if calibrated else "relative contrast (no two opposite anchors)")
reference_index = list(scan_indices).index(SATURATED_SCAN_INDICES[0]) if SATURATED_SCAN_INDICES else 0

focus_setup = None
if FOCUS_PROP_UM != 0:
    if DETECTOR_PIXEL_SIZE_M is None or DETECTOR_DISTANCE_M is None:
        raise ValueError("Set detector pixel size and distance before using nonzero focus")
    focus_setup = {"px_size": DETECTOR_PIXEL_SIZE_M, "ccd_dist": DETECTOR_DISTANCE_M}

recipe = universal.default_universal_phase_retrieval_recipe()
recipe.update(
    projection_model=PROJECTION_MODEL, modes=modes,
    recenter_modal_supports=RECENTER_MODAL_SUPPORTS, mode_support_center=MODE_SUPPORT_CENTER,
    crop=CROP, binning=BINNING, roi=ROI, hologram_intensity_cutoff_vmin=HOLOGRAM_INTENSITY_CUTOFF_VMIN,
    warmup_Nit=WARMUP_NIT, warmup_reference_mode=WARMUP_REFERENCE_MODE, warmup_reference_Nit=WARMUP_REFERENCE_NIT,
    warmup_other_mode=WARMUP_OTHER_MODE, warmup_other_Nit=WARMUP_OTHER_NIT, partial_coherence=PARTIAL_COHERENCE,
    outer_iterations=OUTER_ITERATIONS, inner_mode=INNER_MODE, inner_Nit=INNER_ITERATIONS,
    beta_mode=BETA_MODE, beta_zero=BETA_ZERO, alpha_zero=ALPHA_ZERO, TV_freq=TV_FREQ, average_img=AVERAGE_IMG,
    material_thickness=material, fit_material_thickness=FIT_MATERIAL_THICKNESS,
    physical_iterations=PHYSICAL_ITERATIONS, physical_projection_object_roi=PHYSICAL_PROJECTION_OBJECT_ROI,
    physical_phase_reference=PHYSICAL_PHASE_REFERENCE, saturated_states=saturated_states,
    fit_illumination_intensity=FIT_ILLUMINATION_INTENSITY, align_global_phase=ALIGN_GLOBAL_PHASE,
    fit_aperture_illumination=FIT_APERTURE_ILLUMINATION,
    object_illumination_mode=OBJECT_ILLUMINATION_MODE, aperture_factor_type=APERTURE_FACTOR_TYPE,
    illumination_intensity_reference_observation=reference_index,
    constrain_nonphysical_modes_common=COMMON_SECONDARY_MODES,
    projection_focus_prop_um=FOCUS_PROP_UM, projection_focus_phase_rad=FOCUS_PHASE_RAD,
    projection_focus_setup=focus_setup,
    physical_spatial_weighting=PHYSICAL_SPATIAL_WEIGHTING,
    clip_magnetization=CLIP_MAGNETIZATION, magnetization_trend=MAGNETIZATION_TREND,
    initial_binary_magnetization=INITIAL_BINARY_MAGNETIZATION, binary_magnetization=BINARY_MAGNETIZATION,
    projection_every=len(state_labels), projection_relaxation=PROJECTION_RELAXATION,
    final_projection_relaxation=FINAL_PROJECTION_RELAXATION, final_fourier_constraint=FINAL_FOURIER_CONSTRAINT,
    projection_constraints_inside_support_only=PROJECTION_CONSTRAINTS_INSIDE_SUPPORT_ONLY, zero_magnetization_outside_support=ZERO_MAGNETIZATION_OUTSIDE_SUPPORT,
    fft_workers=FFT_WORKERS, random_seed=RANDOM_SEED,
)
fields, received_start, components, bsmasks, errors = universal.universal_phase_retrieval_algorithm(
    holograms, mask_pixel, support, state_labels=state_labels,
    energy_labels=energies, polarization_coefficients=polarizations,
    illumination_labels=["beam0"]*len(state_labels), universal_recipe=recipe,
    start_fields=warmup.copy(), phase_retrieval_kernel=universal.multimode_phase_retrieval_kernel,
)
np.testing.assert_allclose(received_start, warmup, rtol=1e-14, atol=0.)
assert list(components["state_names"]) == state_labels
print("Physical model log-fit RMS:", components["fit_residual_rms"])

In [ ]:
objects, intensity_residual = workflow.field_diagnostics(fields, holograms, bsmasks, state_labels)
# Library component maps use the shifted fitting frame; center once for display.
magnetization = np.fft.fftshift(components["magnetization"], axes=(-2, -1))
active = material > 0
mean_magnetization = magnetization[:, active].mean(axis=1)
limit = max(1., float(np.max(abs(magnetization[:, active]))))
workflow.plot_stack(np.where(active, magnetization, np.nan), state_labels, "Reduced magnetization" if calibrated else "Relative magnetic contrast", "RdBu_r", -limit, limit)
common_wave = np.fft.fftshift(components["common_exit_waves"], axes=(-2, -1))
workflow.plot_stack(abs(common_wave), ["beam0"], "Shared illumination/reference wave amplitude")
workflow.plot_stack(np.angle(common_wave), ["beam0"], "Shared illumination/reference wave phase", "twilight", -np.pi, np.pi)
print("Charge response:", components["charge_response"])
print("Magnetic response:", components["magnetic_response"])
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(mean_magnetization, "o-"); axes[0].set(xlabel="Observation (acquisition order)", ylabel="Mean magnetization" if calibrated else "Mean relative contrast")
illumination = np.asarray(components.get("illumination_intensity_factors", np.ones(len(state_labels))))
axes[1].plot(illumination, "o-"); axes[1].set(xlabel="Observation", ylabel="Relative illumination intensity")
plt.show()
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with h5py.File(OUTPUT, "w") as h:
    h.attrs["source_file"] = str(INPUT.resolve())
    h.attrs["calibration"] = "two_saturated_endpoints" if calibrated else "relative_contrast"
    h.attrs["component_coordinates"] = "components: solver fitting frame; magnetization: centered display frame"
    for name, array in [("fields", fields), ("warmup_fields", warmup), ("objects", objects),
                        ("holograms", holograms), ("mask_pixel", mask_pixel), ("bsmasks", bsmasks),
                        ("supportmask", support), ("material_thickness", material), ("modes", modes),
                        ("scan_indices", scan_indices), ("energy_eV", energies), ("polarization", polarizations),
                        ("magnetization", magnetization), ("mean_magnetization", mean_magnetization),
                        ("intensity_residual", intensity_residual), ("saturated_scan_indices", SATURATED_SCAN_INDICES),
                        ("saturation_signs", SATURATION_SIGNS)]:
        h.create_dataset(name, data=array, compression="gzip")
    h.create_dataset("state_labels", data=state_labels, dtype=h5py.string_dtype())
    workflow.save_tree(h.create_group("components"), components)
    workflow.save_tree(h.create_group("errors"), errors)
    # Recipe settings and material are saved for reproducibility, not used as a UI.
    workflow.save_tree(h.create_group("recipe"), recipe)
print("Saved", OUTPUT)